### Making the cleaning in our data, after the inspection in the bronze layer

In [0]:
from pyspark.sql import functions as F

path = "/Volumes/sales_lakehouse/bronze/bronze_volume/Product.csv"
product_uncleaned = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("sep", "\t")
    .load(path))


### Making color standarization, to have unique values

In [0]:
from pyspark.sql import functions as F
product_color_std = product_uncleaned.withColumn(
    "Color",
    F.when(F.upper(F.col("Color")) == "MULTI", "Multi")
     .when(F.upper(F.col("Color")) == "RED", "Red")
     .when(F.upper(F.col("Color")) == "YELLOW", "Yellow")
     .when(F.upper(F.col("Color")) == "NA","No Color Available")
     .otherwise(F.col("Color"))
)
product_color_std.select("Color").distinct().show()

### Same for Subcategories

In [0]:
product_subcategory_std = product_color_std.withColumn(
    "Subcategory",
    F.when(F.col("Subcategory") == "Whels", "Wheels")
     .otherwise(F.col("Subcategory"))
)
product_subcategory_std.select("Subcategory").distinct().show()

### Cleaning the $ and commas from the Standard Cost column

In [0]:
from pyspark.sql.types import DecimalType
product_cost_cleaned = product_subcategory_std.withColumn(
    "Standard Cost",
    F.regexp_replace(F.col("Standard Cost"), "[$,]", "")
).withColumn(
    "Standard Cost",
    F.col("Standard Cost").cast(DecimalType(10,2))
)
product_cost_cleaned.select("Standard Cost").show(30, truncate=False)
product_cost_cleaned.printSchema()

### Checking again the duplicates, after the standarizations. 
### See if the duplicated are more now.
### From 7 now i have 8

In [0]:
total_rows = product_cost_cleaned.count()
unique_rows = product_cost_cleaned.distinct().count()
duplicate_count = total_rows - unique_rows
print(f"Total rows: {total_rows}")
print(f"Unique rows: {unique_rows}")
print(f"Total duplicate rows to remove: {duplicate_count}")
product_cost_cleaned.groupBy("ProductKey").count().filter("count > 1").show()

### Dropping duplicates

In [0]:
product_deduplicated = product_cost_cleaned.dropDuplicates()
product_deduplicated.groupBy("ProductKey").count().filter("count > 1").show()

### Making all columns low case for better transormations later

In [0]:
column_mapping = {
    "ProductKey": "product_key",
    "Product": "product",
    "Standard Cost": "standard_cost",
    "Color": "color",
    "Subcategory": "subcategory",
    "Category": "category",
    "Background Color Format": "background_color_format",
    "Font Color Format": "font_color_format"
}

###Changing the table with the values in the dictionary

In [0]:
product_silver = product_deduplicated
for old_name, new_name in column_mapping.items():
    product_silver = product_silver.withColumnRenamed(old_name, new_name)
    print(f"Renamed: {old_name} → {new_name}")

In [0]:
product_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sales_lakehouse.silver.product")
display(product_silver)

In [0]:
%sql
Describe Table sales_lakehouse.silver.product

In [0]:
silver_product = spark.table("sales_lakehouse.silver.product")
silver_sales = spark.table("sales_lakehouse.silver.sales")

missing_from_silver = silver_product.join(silver_sales, "product_key", "left_anti")

print(f"Products without any sale {missing_from_silver.count()}")